# Hunting Hidden Galaxy Collisions with AI
Welcome to the interactive demonstration for our galaxy merger and anomaly detection pipeline! 

This notebook will quickly walk you through downloading a small sample of SDSS data, processing it with a ResNet50 deep learning model, detecting structural anomalies, and scanning for multi-component systems (like mergers) directly in the image plane.

### Step 1: Environment Setup
**If you are running this in Google Colab**, uncomment the code below to clone the repository, install dependencies, and navigate into the project folder.

In [ ]:
# !git clone https://github.com/JohnCassavetes/astro1.git
# %cd astro1
# !pip install -r requirements.txt

### Step 2: Configure for a Fast Demo
By default, the pipeline scales to thousands of images. For this quick 2-minute demo, we will programmatically override the configuration to download only **100** random SDSS galaxies instead of 5,000.

In [ ]:
import yaml

# Load the config file
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Hardcode the limit to 100 for this fast tutorial
config['pipeline']['sdss']['limit'] = 100

# Save it back
with open("config.yaml", "w") as f:
    yaml.dump(config, f)

print("Configuration updated successfully to download 100 images!")

### Step 3: Run the Pipeline Modules
Now we will sequentially run the 5 core modules of the pipeline directly from the command line interface (CLI). Watch the output logs below to understand what each stage is doing!

In [ ]:
print("--- STAGE 1: Downloading 100 SDSS cutouts ---")
!python scripts/download_data.py

print("\n--- STAGE 2: Preprocessing and Normalizing ---")
!python scripts/preprocess_images.py

print("\n--- STAGE 3: Extracting ResNet50 Embeddings ---")
!python scripts/generate_embeddings.py

print("\n--- STAGE 4: Isolation Forest Anomaly Detection ---")
!python scripts/detect_anomalies.py

print("\n--- STAGE 5: Scanning Raw Images for Mergers ---")
!python scripts/scan_raw_secondary_sources.py

### Step 4: Visualize the Results
The scanner script (Stage 5) automatically drew red bounding boxes around the primary galaxies and cyan bounding boxes around any high-confidence secondary/multi-component galaxies it found physically nearby.

Let's visualize any successful candidates!

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image

# Find all generated overlay diagnostic images
overlay_dir = "results/final/raw_object_scan/overlays"
overlays = glob.glob(os.path.join(overlay_dir, "*.png"))

if not overlays:
    print("Out of 100 random galaxies, no strong merger candidates were found in this batch!")
    print("Try increasing the limit in config.yaml to 500 and running again.")
else:
    print(f"Found {len(overlays)} strong candidates! Plotting them below:")
    
    # Plot up to 4 images max
    num_to_plot = min(len(overlays), 4)
    fig, axes = plt.subplots(1, num_to_plot, figsize=(15, 5))
    
    # Handle single image case
    if num_to_plot == 1:
        axes = [axes]
        
    for ax, img_path in zip(axes, overlays[:num_to_plot]):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        # Set title to objid
        ax.set_title(os.path.basename(img_path).split('_')[0])
        
    plt.tight_layout()
    plt.show()